# encoder
# decoder
# train
# generate

In [10]:
import torch
import torch.nn as nn

In [11]:
chars = "0123456789-/"
special_tokens = ["<SOS>", "<EOS>"]

itos = special_tokens + list(chars)
stoi = {ch: i for i, ch in enumerate(itos)}

SOS_token = stoi["<SOS>"]
EOS_token = stoi["<EOS>"]

vocab_size = len(itos)

print(itos)
print(stoi)

['<SOS>', '<EOS>', '0', '1', '2', '3', '4', '5', '6', '7', '8', '9', '-', '/']
{'<SOS>': 0, '<EOS>': 1, '0': 2, '1': 3, '2': 4, '3': 5, '4': 6, '5': 7, '6': 8, '7': 9, '8': 10, '9': 11, '-': 12, '/': 13}


In [ ]:
class EncoderRNN(nn.Module):
    def __init__(self, vocab_size, hidden_size):
        super().__init__()
        self.hidden_size = hidden_size
        self.embedding = nn.Embedding(vocab_size, hidden_size)
        self.rnn = nn.RNN(hidden_size, hidden_size, batch_first=True)
        self.dropout = nn.Dropout(0.1)

        # 你来写 embedding
        # 你来写 rnn

    def forward(self, input):
        # 要限定 input的shape
        # input.shape (batch_size, seqlen)
        # 你来写一次前向传播
        input = self.embedding(input)
        input = self.dropout(input)
        # hidden不受 batch_first=True的控制，其shape仍然为(direction*num,batch_size,hidden_size)
        # output.shape 是 (batch_size,seq_len,direction*hidden_size)
        output, hidden = self.rnn(input)
        return output, hidden


In [13]:
import random
def make_sample():
    ep = 100
    inputs = []
    targets = []
    for i in range(ep):
        year = random.randint(1950, 2050)
        day = random.randint(1, 28)
        month = random.randint(1, 12)
        input = f"{year}-{month}-{day}"
        inputs.append(input)
        target = f"{day}/{month}/{year}"
        targets.append(target)
    return inputs, targets

inputs, targets = make_sample()
print(inputs[0:10])
print(targets[0:10])


['2024-5-26', '2048-3-11', '2002-3-4', '2005-9-2', '2027-3-12', '1959-10-13', '2039-12-16', '2022-8-18', '2028-1-14', '2016-11-14']
['26/5/2024', '11/3/2048', '4/3/2002', '2/9/2005', '12/3/2027', '13/10/1959', '16/12/2039', '18/8/2022', '14/1/2028', '14/11/2016']


In [14]:
def tensor_from_string(s, add_eos=False):
    input_tensor = torch.tensor([stoi[ch] for ch in s], dtype=torch.long)
    if add_eos:
        new_value = torch.tensor([EOS_token])
        input_tensor = torch.cat([input_tensor, new_value],dim=0)
    return input_tensor.unsqueeze(0)

def string_from_tensor(tensor):
    s = ''
    tensor = tensor.squeeze(0)
    for token in tensor:
        idx = token.item()
        if idx != EOS_token:
            s += itos[idx]
    return s

In [15]:

x = tensor_from_string("2002-1-23")
print(x, x.shape)
y = tensor_from_string("23/1/2002", add_eos=True)
print(y, y.shape)


tensor([[ 4,  2,  2,  4, 12,  3, 12,  4,  5]]) torch.Size([1, 9])
tensor([[ 4,  5, 13,  3, 13,  4,  2,  2,  4,  1]]) torch.Size([1, 10])


In [16]:
print(string_from_tensor(x))
print(string_from_tensor(y))

2002-1-23
23/1/2002


In [17]:
hidden_size = 5
encoder = EncoderRNN(vocab_size, hidden_size)

x = tensor_from_string("2002-1-23")
print("x", x.shape)

encoder_output, encoder_hidden = encoder.forward(x);
print("encoder_output:", encoder_output.shape)
print("encoder_hidden:", encoder_hidden.shape)

x torch.Size([1, 9])
encoder_output: torch.Size([1, 9, 5])
encoder_hidden: torch.Size([1, 1, 5])


In [ ]:
class DecoderRNN(nn.Module):
    def __init__(self, vocab_size, hidden_size, output_size):
        super().__init__()
        self.hidden_size = hidden_size
        self.embedding = nn.Embedding(vocab_size, hidden_size)
        self.rnn = nn.RNN(hidden_size, hidden_size, batch_first=True)
        self.out = nn.Linear(hidden_size, output_size)

    def forward(self, encoder_output, target_tensor):
        # encoder_output是encoder输出的hiddeng向量
        # target_tensor是decoder输出的向量
        hidden = encoder_output
        input_token = SOS_token #设置起始词元
        outputs = []
        input_token = torch.full((1, 1), SOS_token, dtype=torch.long)
        for i in range(12):
            predictIdx, hidden = self.forward_step(hidden, input_token)
            outputs.append(predictIdx)

            if target_tensor is not None:
                input_token[0][0] = target_tensor[i:i+1]
            if target_tensor is None:
                input_token[0][0] = predictIdx
            
        return torch.cat(outputs, dim=1)  # (batch, max_len)
        
    def forward_step(self, hidden, input_token):
        # input_token.shape是(batch_size,seq_len),元素内容是词元id
        # embedded.shape应当是(batch_size,seq_len,hidden)，元素内容是次元id对应的embedding值
        embedded = self.embedding(input_token)
        # rnn_output 是所有时间步骤的最后一层,shape是 (batch_size,seq_len,hidden)
        # hidden 是最后一个时间步骤的所有层,shape是 (batch_size, num_layers * num_directions,hidden),本例中 num_layers * num_directions 是1
        rnn_output, hidden = self.rnn(embedded, hidden) 
        output = self.out(hidden[-1]) # out 的输入是最后一维是hidden的任何形状，输出是将输入的最后一维改成和Linear初始化设置的输出一样的形状
        predictIdx = output.argmax(dim = -1, keepdim=True)
        return predictIdx, hidden

    

In [8]:
hidden_size = 5
decoder = DecoderRNN(vocab_size, hidden_size, vocab_size)


In [10]:
print(decoder.out.out_features)

14
